# Cross-Attention Fusion Experiments

Here I train the two camera–LiDAR fusion models:

Standard fusion model — trained with clean camera and clean LiDAR inputs
Adversarially trained fusion model — trained with a 50/50 mix of clean camera inputs and FGSM-attacked camera inputs, while LiDAR remains clean

## 1. Install & Imports

In [ ]:
# Install pykitti and import the libraries used in this notebook
!pip install pykitti --quiet

import torch, torch.nn as nn, os, zipfile, urllib.request
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# Remove incomplete KITTI zip files if a previous download failed
import os, glob

# Delete any incomplete zip downloads
for f in glob.glob('/content/kitti/*.zip'):
    print("Removing partial zip:", f)
    os.remove(f)

## 2. Download KITTI Drives

In [ ]:
# Download the KITTI raw drives used in this experiment
BASE_DIR = '/content/kitti'
os.makedirs(BASE_DIR, exist_ok=True)

def download_kitti_drive(date, drive):
    drive_str = f'{date}_drive_{drive}_sync'
    base_url  = 'https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data'
    url       = f'{base_url}/{date}_drive_{drive}/{drive_str}.zip'
    zip_path  = f'{BASE_DIR}/{drive_str}.zip'
    if os.path.exists(f'{BASE_DIR}/{date}/{drive_str}'):
        print(f'  skip {drive_str}'); return
    print(f'  downloading {drive_str}...', end=' ')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(BASE_DIR)
    os.remove(zip_path); print('done.')

def download_kitti_calib(date):
    base_url  = 'https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data'
    url       = f'{base_url}/{date}_calib.zip'
    zip_path  = f'{BASE_DIR}/{date}_calib.zip'
    calib_dir = f'{BASE_DIR}/{date}'
    if os.path.exists(calib_dir) and any('calib' in f for f in os.listdir(calib_dir)):
        print(f'  calibration ready'); return
    os.makedirs(calib_dir, exist_ok=True)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(BASE_DIR)
    os.remove(zip_path); print('  calibration downloaded')

# Initial drive lists before the controlled stratified split

TRAIN_DRIVES = [
    ('2011_09_26','0005'),
    ('2011_09_26','0009'),
    ('2011_09_26','0011'),
    ('2011_09_26','0013'),
    ('2011_09_26','0014'),
    ('2011_09_26','0015'),
    ('2011_09_26','0017'),
    ('2011_09_26','0018'),
    ('2011_09_26','0019'),
    ('2011_09_26','0020'),
]

TEST_DRIVES = [
    ('2011_09_26','0001'),
    ('2011_09_26','0002'),
    ('2011_09_26','0029'),
    ('2011_09_26','0032'),
]

print('Downloading calibration...')
download_kitti_calib('2011_09_26')
print('\nDownloading training drives...')
for date, drive in TRAIN_DRIVES:
    try:    download_kitti_drive(date, drive)
    except: print(f'  skipped {drive} (not available)')
print('\nDownloading test drives...')
for date, drive in TEST_DRIVES:
    download_kitti_drive(date, drive)
print('\nAll drives ready.')


## 3. Load Camera + LiDAR Data

Loads RGB camera frames, LiDAR point clouds, and OxTS motion data. The LiDAR point cloud is converted into eight real-world angular sectors so the attention weights are easier to interpret.

In [ ]:
# Load KITTI frames and convert LiDAR clouds into angular sector tokens
import pykitti

LABEL_MAP   = {'straight':0,'turn_left':1,'turn_right':2,'stop':3}
LABEL_NAMES = ['straight','turn_left','turn_right','stop']
NUM_CLASSES = 4

NUM_LIDAR_SECTORS = 8
PTS_PER_SECTOR = 128

SECTOR_NAMES = [
    "front",
    "front_left",
    "left",
    "rear_left",
    "rear",
    "rear_right",
    "right",
    "front_right"
]

def get_action_label(oxts_packet):
    vf = oxts_packet.packet.vf
    wz = oxts_packet.packet.wz
    if   vf < 1.0:    return 'stop'
    elif wz >  0.03:  return 'turn_left'
    elif wz < -0.03:  return 'turn_right'
    else:             return 'straight'

def normalize_lidar_points(points):
    points = points.copy()
    points[:, 0] = np.clip(points[:, 0], -50, 50) / 50.0
    points[:, 1] = np.clip(points[:, 1], -50, 50) / 50.0
    points[:, 2] = np.clip(points[:, 2], -5, 5) / 5.0
    return points.astype(np.float32)


def lidar_to_angular_sectors(cloud, points_per_sector=PTS_PER_SECTOR):
    pts = cloud[:, :3].astype(np.float32)
    pts = pts[np.isfinite(pts).all(axis=1)]

    dist = np.linalg.norm(pts[:, :2], axis=1)
    pts = pts[(dist > 1.0) & (dist < 80.0)]

    if len(pts) == 0:
        return np.zeros((NUM_LIDAR_SECTORS, points_per_sector, 3), dtype=np.float32)

    angles = np.arctan2(pts[:, 1], pts[:, 0])

    sector_ids = np.floor(
        ((angles + np.pi / NUM_LIDAR_SECTORS) % (2 * np.pi))
        / (2 * np.pi / NUM_LIDAR_SECTORS)
    ).astype(np.int64)

    sectors = np.zeros((NUM_LIDAR_SECTORS, points_per_sector, 3), dtype=np.float32)

    for s in range(NUM_LIDAR_SECTORS):
        sector_pts = pts[sector_ids == s]

        if len(sector_pts) == 0:
            continue

        replace = len(sector_pts) < points_per_sector
        idx = np.random.choice(len(sector_pts), points_per_sector, replace=replace)
        sampled = sector_pts[idx]

        sectors[s] = normalize_lidar_points(sampled)

    return sectors

def load_drive(date, drive, fraction=1.0):
    data = pykitti.raw(BASE_DIR, date, drive)
    images, lidars, labels = [], [], []
    for img, velo, oxts in zip(data.cam2, data.velo, data.oxts):
        images.append(img)
        lidars.append(lidar_to_angular_sectors(velo))
        labels.append(LABEL_MAP[get_action_label(oxts)])
    cutoff = int(len(images) * fraction)
    images, lidars, labels = images[:cutoff], lidars[:cutoff], labels[:cutoff]
    dist = dict(Counter([LABEL_NAMES[l] for l in labels]))
    print(f'  drive {drive}: {len(images)} frames | {dist}')
    return images, lidars, labels

print('Loading TRAINING drives at 50%...')
train_images, train_lidars, train_labels = [], [], []
for date, drive in TRAIN_DRIVES:
    path = f'{BASE_DIR}/{date}/{date}_drive_{drive}_sync'
    if not os.path.exists(path):
        print(f'  drive {drive} not on disk, skipping'); continue
    imgs, lids, lbls = load_drive(date, drive, fraction=0.5)
    train_images.extend(imgs); train_lidars.extend(lids); train_labels.extend(lbls)

print('\nLoading TEST drives at 50%...')
test_images, test_lidars, test_labels = [], [], []
for date, drive in TEST_DRIVES:
    path = f'{BASE_DIR}/{date}/{date}_drive_{drive}_sync'
    if not os.path.exists(path):
        print(f'  drive {drive} not on disk, skipping'); continue
    imgs, lids, lbls = load_drive(date, drive, fraction=0.5)
    test_images.extend(imgs); test_lidars.extend(lids); test_labels.extend(lbls)

print(f'\nTraining: {len(train_images)} frames')
print(f'  Distribution: {dict(Counter([LABEL_NAMES[l] for l in train_labels]))}')
print(f'Test: {len(test_images)} frames')
print(f'  Distribution: {dict(Counter([LABEL_NAMES[l] for l in test_labels]))}')

# Controlled stratified split used for the final fair comparison
# This keeps all four classes represented in train and test sets.

from sklearn.model_selection import train_test_split

all_images = list(train_images) + list(test_images)
all_lidars = list(train_lidars) + list(test_lidars)
all_labels = list(train_labels) + list(test_labels)

all_indices = np.arange(len(all_labels))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=all_labels
)

train_images = [all_images[i] for i in train_idx]
train_lidars = [all_lidars[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

test_images = [all_images[i] for i in test_idx]
test_lidars = [all_lidars[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

print("\nCONTROLLED STRATIFIED SPLIT APPLIED")
print("-" * 45)
print(f"Training: {len(train_images)} frames")
print("  Distribution:", dict(Counter([LABEL_NAMES[x] for x in train_labels])))
print(f"Test: {len(test_images)} frames")
print("  Distribution:", dict(Counter([LABEL_NAMES[x] for x in test_labels])))

## 4. Dataset and DataLoaders

A mild square-root weighted sampler is used to reduce class imbalance without over-amplifying minority classes.

In [ ]:
# Build datasets and use a mild sampler to handle class imbalance
from torch.utils.data import WeightedRandomSampler

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomCrop((224,224)),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
TRANSFORM_TEST = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
UNNORMALIZE = transforms.Normalize(
    mean=[-0.485/0.229,-0.456/0.224,-0.406/0.225],
    std=[1/0.229,1/0.224,1/0.225]
)

class KITTIMultiModalDataset(Dataset):
    def __init__(self,images,lidars,labels,transform=None):
        self.images=images; self.lidars=lidars
        self.labels=labels; self.transform=transform
    def __len__(self): return len(self.images)
    def __getitem__(self,idx):
        img = self.images[idx].convert('RGB')
        if self.transform: img=self.transform(img)
        return (img,
                torch.tensor(self.lidars[idx],dtype=torch.float32),
                torch.tensor(self.labels[idx],dtype=torch.long))

train_dataset = KITTIMultiModalDataset(train_images,train_lidars,train_labels,TRANSFORM_TRAIN)
test_dataset  = KITTIMultiModalDataset(test_images, test_lidars, test_labels, TRANSFORM_TEST)

# Mild sqrt sampler:
# This keeps the sampler consistent with the camera-only experiment.
# It avoids straight-only collapse without over-amplifying minority classes.

BATCH_SIZE = 16

train_counts = Counter(train_labels)

sample_weights = [
    1.0 / np.sqrt(train_counts[int(label)])
    for label in train_labels
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Train batches: {len(train_loader)} ({len(train_dataset)} samples)')
print(f'Test  batches: {len(test_loader)} ({len(test_dataset)} samples)')
si,sl,sb = next(iter(train_loader))
print(f'Image batch: {si.shape}')
print(f'LiDAR batch: {sl.shape}  <- should be (batch, 8, 128, 3)')
print(f'Label batch : {sb.shape}')
print(f'LiDAR sectors: {SECTOR_NAMES}')

## 5. Cross-Attention Fusion Model

The camera feature acts as the query, and the eight LiDAR sector tokens act as keys and values.

In [ ]:
# Define the fusion model used by both training procedures
class CrossAttentionFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256, num_heads=4, n_lidar_tokens=8):
        super().__init__()
        self.n_lidar_tokens = n_lidar_tokens

        # Camera branch
        resnet = models.resnet18(weights='IMAGENET1K_V1')
        self.image_backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.img_proj = nn.Linear(512, embed_dim)

        # Shared PointNet-style MLP for each real-world LiDAR sector
        self.lidar_mlp = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim),
            nn.ReLU()
        )

        # Camera token queries 8 LiDAR sector tokens
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=False
        )

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, image_input, lidar_input, return_attn=False):
        """
        image_input: (B, 3, 224, 224)
        lidar_input: (B, 8, PTS_PER_SECTOR, 3)
        """
        B, S, P, C = lidar_input.shape

        # Camera feature
        img_feat = torch.flatten(self.image_backbone(image_input), 1)
        img_seq = self.img_proj(img_feat).unsqueeze(0)  # (1, B, 256)

        # LiDAR sector tokens
        lidar_flat = lidar_input.view(B * S, P, C)      # (B*8, P, 3)
        point_feats = self.lidar_mlp(lidar_flat)        # (B*8, P, 256)

        sector_tokens = point_feats.mean(dim=1)         # (B*8, 256)
        sector_tokens = sector_tokens.view(B, S, -1)    # (B, 8, 256)

        lidar_seq = sector_tokens.permute(1, 0, 2)      # (8, B, 256)

        fused, attn_weights = self.cross_attention(
            query=img_seq,
            key=lidar_seq,
            value=lidar_seq
        )

        logits = self.classifier(fused.squeeze(0))

        if return_attn:
            return logits, attn_weights

        return logits


def make_model():
    return CrossAttentionFusionModel(
        num_classes=NUM_CLASSES,
        embed_dim=256,
        num_heads=4,
        n_lidar_tokens=NUM_LIDAR_SECTORS
    ).to(device)


def make_optimizer(model):
    return torch.optim.AdamW([
        {'params': model.image_backbone.parameters(), 'lr': 1e-5},
        {'params': model.lidar_mlp.parameters(),      'lr': 5e-4},
        {'params': model.img_proj.parameters(),       'lr': 5e-4},
        {'params': model.cross_attention.parameters(),'lr': 5e-4},
        {'params': model.classifier.parameters(),     'lr': 5e-4},
    ], weight_decay=1e-4)


def reset_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Same architecture and same initialization seed for fair comparison
reset_seed(42)
standard_model = make_model()

reset_seed(42)
adv_model = make_model()

total_p = sum(p.numel() for p in standard_model.parameters())
print(f'Model architecture: {total_p:,} parameters each')

# Sanity check
with torch.no_grad():
    di = torch.randn(4, 3, 224, 224).to(device)
    dl = torch.randn(4, NUM_LIDAR_SECTORS, PTS_PER_SECTOR, 3).to(device)

    out, attn = standard_model(di, dl, return_attn=True)

    print(f'Output: {out.shape}')
    print(f'Attn : {attn.shape} <- should be (4, 1, 8)')
    print(f'Attn sums to 1: {attn[0,0].sum():.4f}')
    print(f'Sectors: {SECTOR_NAMES}')

## 6. FGSM Attack Function

In [ ]:
# FGSM is applied only to normalized camera images
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

NORM_LOWER = (0 - IMAGENET_MEAN) / IMAGENET_STD
NORM_UPPER = (1 - IMAGENET_MEAN) / IMAGENET_STD

def clamp_normalized_image(x):
    return torch.max(torch.min(x, NORM_UPPER), NORM_LOWER)

def fgsm_attack_camera_only(model, images, lidars, labels, epsilon):
    """
    Perturb only the camera input.
    LiDAR remains clean.
    Correctly handles ImageNet-normalized images.
    """
    was_training = model.training
    model.eval()

    images_adv = images.clone().detach().requires_grad_(True)
    lidars = lidars.clone().detach()
    labels = labels.clone().detach()

    logits = model(images_adv, lidars)
    loss = nn.CrossEntropyLoss()(logits, labels)

    model.zero_grad(set_to_none=True)
    loss.backward()

    adv = images_adv + epsilon * images_adv.grad.sign()
    adv = clamp_normalized_image(adv).detach()

    if was_training:
        model.train()

    return adv

print('FGSM camera-only attack ready.')
print('Perturbs camera input only — LiDAR remains clean.')

## 7. Loss Function and Evaluation Helper

Defines cross-entropy loss and an evaluation function that reports raw accuracy, balanced accuracy, and per-class behavior.

In [ ]:
# Use balanced accuracy so minority classes matter during checkpointing
counts = Counter(train_labels)
print("Training label distribution:")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name:>10}: {counts.get(i, 0)}")

criterion = nn.CrossEntropyLoss()
print("Using normal CrossEntropyLoss with mild sqrt WeightedRandomSampler.")


def eval_model(model, loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    class_correct = np.zeros(NUM_CLASSES)
    class_total = np.zeros(NUM_CLASSES)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lids, lbls in loader:
            imgs, lids, lbls = imgs.to(device), lids.to(device), lbls.to(device)

            logits = model(imgs, lids)
            loss = criterion(logits, lbls)
            preds = logits.argmax(1)

            total_loss += loss.item() * len(lbls)
            correct += (preds == lbls).sum().item()
            total += len(lbls)

            for c in range(NUM_CLASSES):
                mask = (lbls == c)
                class_total[c] += mask.sum().item()
                class_correct[c] += (preds[mask] == lbls[mask]).sum().item()

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(lbls.cpu().numpy().tolist())

    val_acc = correct / total

    per_class_acc = np.divide(
        class_correct,
        class_total,
        out=np.zeros_like(class_correct),
        where=class_total > 0
    )

    balanced_acc = per_class_acc.mean()

    return total_loss / total, val_acc, balanced_acc, per_class_acc, all_preds, all_labels


print("Loss function and balanced evaluation helper ready.")

## 8. Train Standard Fusion Model

In [ ]:
# Train the standard fusion model on clean inputs
NUM_EPOCHS = 15

std_optimizer = make_optimizer(standard_model)
std_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    std_optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

std_history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_bal_acc': []
}

print(f'Training STANDARD model for {NUM_EPOCHS} epochs (clean data only)...')
print('Epoch | Train Loss | Train Acc | Val Loss | Val Acc | Bal Acc')
print('-' * 72)

best_std_bal = 0.0
best_std_epoch = 0

for epoch in range(1, NUM_EPOCHS + 1):
    standard_model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for imgs, lids, lbls in train_loader:
        imgs, lids, lbls = imgs.to(device), lids.to(device), lbls.to(device)

        std_optimizer.zero_grad(set_to_none=True)

        logits = standard_model(imgs, lids)
        loss = criterion(logits, lbls)

        loss.backward()
        std_optimizer.step()

        train_loss_sum += loss.item() * len(lbls)
        train_correct += (logits.argmax(1) == lbls).sum().item()
        train_total += len(lbls)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    val_loss, val_acc, val_bal_acc, per_class_acc, val_preds, val_labels = eval_model(
        standard_model,
        test_loader
    )

    std_scheduler.step()

    # Save best model using balanced accuracy, not raw validation accuracy
    if val_bal_acc > best_std_bal:
        best_std_bal = val_bal_acc
        best_std_epoch = epoch
        torch.save(standard_model.state_dict(), '/content/best_standard_model.pth')

    std_history['train_loss'].append(train_loss)
    std_history['train_acc'].append(train_acc)
    std_history['val_loss'].append(val_loss)
    std_history['val_acc'].append(val_acc)
    std_history['val_bal_acc'].append(val_bal_acc)

    print(
        f'{epoch:>5} | '
        f'{train_loss:>10.4f} | '
        f'{train_acc:>8.2%} | '
        f'{val_loss:>8.4f} | '
        f'{val_acc:>7.2%} | '
        f'{val_bal_acc:>7.2%}'
    )

    if epoch == 1 or epoch % 5 == 0:
        pred_names = [LABEL_NAMES[p] for p in val_preds]
        true_names = [LABEL_NAMES[y] for y in val_labels]

        print("  Val prediction distribution:", dict(Counter(pred_names)))
        print("  Val true distribution      :", dict(Counter(true_names)))

        print("  Per-class val accuracy:")
        for i, name in enumerate(LABEL_NAMES):
            print(f"    {name:>10}: {per_class_acc[i]:.2%}")

print(f'\nStandard model done.')
print(f'Best balanced val accuracy: {best_std_bal:.2%}')
print(f'Best standard epoch: {best_std_epoch}')

standard_model.load_state_dict(torch.load('/content/best_standard_model.pth'))
print('Best standard weights loaded.')

## 9. Train Adversarially Trained Fusion Model

In [ ]:
# Train the adversarial fusion model with half clean and half attacked camera inputs
ADV_TRAIN_EPS = 0.05
ADV_RATIO = 0.5

adv_optimizer = make_optimizer(adv_model)
adv_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    adv_optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

adv_history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_bal_acc': []
}

print(f'Training ADVERSARIALLY TRAINED model for {NUM_EPOCHS} epochs...')
print(f'Each batch: {int(ADV_RATIO * 100)}% adversarial camera + {int((1 - ADV_RATIO) * 100)}% clean camera')
print(f'LiDAR stays clean for all samples.')
print(f'Attack epsilon during training: {ADV_TRAIN_EPS}')
print('Epoch | Train Loss | Train Acc | Val Loss | Val Acc | Bal Acc')
print('-' * 72)

best_adv_bal = 0.0
best_adv_epoch = 0

for epoch in range(1, NUM_EPOCHS + 1):
    adv_model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for imgs, lids, lbls in train_loader:
        imgs, lids, lbls = imgs.to(device), lids.to(device), lbls.to(device)

        batch_size = imgs.size(0)
        n_adv = int(batch_size * ADV_RATIO)

        mixed_imgs = imgs.clone().detach()

        if n_adv > 0:
            idx = torch.randperm(batch_size, device=device)[:n_adv]

            adv_imgs = fgsm_attack_camera_only(
                adv_model,
                imgs[idx],
                lids[idx],
                lbls[idx],
                epsilon=ADV_TRAIN_EPS
            )

            mixed_imgs[idx] = adv_imgs

        adv_optimizer.zero_grad(set_to_none=True)

        logits = adv_model(mixed_imgs, lids)
        loss = criterion(logits, lbls)

        loss.backward()
        adv_optimizer.step()

        train_loss_sum += loss.item() * batch_size
        train_correct += (logits.argmax(1) == lbls).sum().item()
        train_total += batch_size

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    val_loss, val_acc, val_bal_acc, per_class_acc, val_preds, val_labels = eval_model(
        adv_model,
        test_loader
    )

    adv_scheduler.step()

    # Save best model using balanced accuracy, not raw validation accuracy.
    if val_bal_acc > best_adv_bal:
        best_adv_bal = val_bal_acc
        best_adv_epoch = epoch
        torch.save(adv_model.state_dict(), '/content/best_adv_model.pth')

    adv_history['train_loss'].append(train_loss)
    adv_history['train_acc'].append(train_acc)
    adv_history['val_loss'].append(val_loss)
    adv_history['val_acc'].append(val_acc)
    adv_history['val_bal_acc'].append(val_bal_acc)

    print(
        f'{epoch:>5} | '
        f'{train_loss:>10.4f} | '
        f'{train_acc:>8.2%} | '
        f'{val_loss:>8.4f} | '
        f'{val_acc:>7.2%} | '
        f'{val_bal_acc:>7.2%}'
    )

    if epoch == 1 or epoch % 5 == 0:
        pred_names = [LABEL_NAMES[p] for p in val_preds]
        true_names = [LABEL_NAMES[y] for y in val_labels]

        print("  Val prediction distribution:", dict(Counter(pred_names)))
        print("  Val true distribution      :", dict(Counter(true_names)))

        print("  Per-class val accuracy:")
        for i, name in enumerate(LABEL_NAMES):
            print(f"    {name:>10}: {per_class_acc[i]:.2%}")

print(f'\nAdversarial model done.')
print(f'Best balanced val accuracy: {best_adv_bal:.2%}')
print(f'Best adversarial epoch: {best_adv_epoch}')

adv_model.load_state_dict(torch.load('/content/best_adv_model.pth'))
print('Best adversarial weights loaded.')

## 10. Training Curves Comparison

Compares clean validation behavior for the standard fusion model and adversarially trained fusion model.

In [ ]:
# Compare training curves for the two fusion models
fig, axes = plt.subplots(1,2,figsize=(14,5))
ep = range(1,NUM_EPOCHS+1)

# Loss
axes[0].plot(ep,std_history['val_loss'],'b-o',label='Standard val loss',  markersize=4)
axes[0].plot(ep,adv_history['val_loss'],'r-o',label='Adversarial val loss',markersize=4)
axes[0].set_title('Validation Loss',fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(ep,[a*100 for a in std_history['val_acc']],'b-o',label='Standard val acc',  markersize=4)
axes[1].plot(ep,[a*100 for a in adv_history['val_acc']],'r-o',label='Adversarial val acc',markersize=4)
axes[1].set_title('Validation Accuracy (%)',fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim([0,100])

plt.suptitle('Standard vs Adversarially Trained Fusion Model',
             fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_comparison.png',dpi=150,bbox_inches='tight')
plt.show()
print('Note: adversarial model may have lower clean val accuracy')
print('but should be more robust under attack — check Cell 11.')


## 11. Conflict Experiment for Both Fusion Models

Evaluates both fusion models across FGSM epsilon values

In [ ]:
# Evaluate the sensor-conflict setting across multiple attack strengths
EPSILONS = [0.0,0.01,0.03,0.05,0.1,0.2,0.3]

std_clean    = {}; std_conflict    = {}
adv_clean    = {}; adv_conflict    = {}

standard_model.eval(); adv_model.eval()

print('Running conflict experiment on BOTH models...\n')
print(f'{"Eps":>5} | {"Std Clean":>10} | {"Std Conflict":>13} | {"Adv Clean":>10} | {"Adv Conflict":>13} | {"Improvement":>11}')
print('-'*75)

for eps in EPSILONS:
    sc=0; sa=0; ac=0; aa=0; total=0
    for imgs,lids,lbls in test_loader:
        imgs,lids,lbls=imgs.to(device),lids.to(device),lbls.to(device)

        # Standard model
        with torch.no_grad():
            sc+=(standard_model(imgs,lids).argmax(1)==lbls).sum().item()
        adv_imgs=fgsm_attack_camera_only(standard_model,imgs,lids,lbls,eps) if eps>0 else imgs
        with torch.no_grad():
            sa+=(standard_model(adv_imgs,lids).argmax(1)==lbls).sum().item()

        # Adversarially trained model
        with torch.no_grad():
            ac+=(adv_model(imgs,lids).argmax(1)==lbls).sum().item()
        adv_imgs2=fgsm_attack_camera_only(adv_model,imgs,lids,lbls,eps) if eps>0 else imgs
        with torch.no_grad():
            aa+=(adv_model(adv_imgs2,lids).argmax(1)==lbls).sum().item()

        total+=len(lbls)

    sc/=total; sa/=total; ac/=total; aa/=total
    improvement = aa - sa
    std_clean[eps]=sc; std_conflict[eps]=sa
    adv_clean[eps]=ac; adv_conflict[eps]=aa

    print(f'{eps:>5.2f} | {sc:>9.2%} | {sa:>12.2%} | {ac:>9.2%} | {aa:>12.2%} | {improvement:>+10.2%}')

print('\nImprovement = Adv Conflict - Std Conflict')
print('Positive = adversarial training helped under sensor conflict')


## 12. Standard Fusion vs. Adversarially Trained Fusion

In [ ]:
# Compares standard fusion vs adversarially trained fusion only

sc_l = [std_clean[e]*100    for e in EPSILONS]
sa_l = [std_conflict[e]*100 for e in EPSILONS]
ac_l = [adv_clean[e]*100    for e in EPSILONS]
aa_l = [adv_conflict[e]*100 for e in EPSILONS]

fig, ax = plt.subplots(figsize=(12,6))

ax.plot(EPSILONS, sc_l, 'b--s', lw=2,   ms=7,
        label='Standard Fusion (both clean)')
ax.plot(EPSILONS, sa_l, 'g-^',  lw=2,   ms=7,
        label='Standard Fusion (conflict scenario)')
ax.plot(EPSILONS, ac_l, 'b:s',  lw=1.5, ms=6,
        label='Adv Trained Fusion (both clean)')
ax.plot(EPSILONS, aa_l, 'm-D',  lw=2.5, ms=9,
        label='Adv Trained Fusion (conflict scenario)')
ax.axhline(25, color='gray', ls=':', lw=1.5, label='Random chance (25%)')

# Shade robustness gain from adversarial training
ax.fill_between(EPSILONS, sa_l, aa_l,
    where=[a >= s for a, s in zip(aa_l, sa_l)],
    alpha=0.15, color='purple',
    label='Robustness gain from adversarial training')

ax.set_xlabel('Epsilon (perturbation strength)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title(
    'Effect of Adversarial Training on Cross-Attention Fusion Robustness\n'
    'Under Sensor Conflict: Adversarial Camera + Clean LiDAR',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10, loc='lower left')
ax.set_ylim([0, 105])
ax.grid(alpha=0.3)

for e, a in zip(EPSILONS, aa_l):
    ax.annotate(f'{a:.1f}%', (e, a),
                textcoords='offset points', xytext=(0,10),
                ha='center', fontsize=8, color='purple')

plt.tight_layout()
plt.savefig('/content/main_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved: /content/main_comparison.png')
print()
print('Key numbers at epsilon=0.05:')
eps_idx = EPSILONS.index(0.05)
print(f'  Standard fusion conflict:    {sa_l[eps_idx]:.2f}%')
print(f'  Adv trained fusion conflict: {aa_l[eps_idx]:.2f}%')
print(f'  Improvement:                 +{aa_l[eps_idx]-sa_l[eps_idx]:.2f}pp')
print()
print('Purple line above green line = adversarial training improved robustness')

## 13. Per-Class Comparison at epsilon = 0.05

In [ ]:
# Compare class-level performance at the main attack strength
def per_class_conflict(model, loader, epsilon):
    model.eval()

    clean_correct = np.zeros(NUM_CLASSES)
    attack_correct = np.zeros(NUM_CLASSES)
    total = np.zeros(NUM_CLASSES)

    for imgs, lids, lbls in loader:
        imgs, lids, lbls = imgs.to(device), lids.to(device), lbls.to(device)

        adv_imgs = fgsm_attack_camera_only(model, imgs, lids, lbls, epsilon)

        with torch.no_grad():
            clean_preds = model(imgs, lids).argmax(1)
            attack_preds = model(adv_imgs, lids).argmax(1)

        for c in range(NUM_CLASSES):
            mask = (lbls == c)
            total[c] += mask.sum().item()
            clean_correct[c] += (clean_preds[mask] == lbls[mask]).sum().item()
            attack_correct[c] += (attack_preds[mask] == lbls[mask]).sum().item()

    clean_acc = np.divide(clean_correct, total, out=np.zeros_like(clean_correct), where=total > 0)
    attack_acc = np.divide(attack_correct, total, out=np.zeros_like(attack_correct), where=total > 0)

    return clean_acc, attack_acc, total

EPS = 0.05

std_clean_pc, std_attack_pc, class_totals = per_class_conflict(
    standard_model, test_loader, EPS
)

adv_clean_pc, adv_attack_pc, _ = per_class_conflict(
    adv_model, test_loader, EPS
)

x = np.arange(NUM_CLASSES)
w = 0.22

fig, ax = plt.subplots(figsize=(13, 6))

b1 = ax.bar(x - 1.5*w, std_clean_pc * 100,  w, label='Standard Fusion: Clean')
b2 = ax.bar(x - 0.5*w, std_attack_pc * 100, w, label='Standard Fusion: Conflict')
b3 = ax.bar(x + 0.5*w, adv_clean_pc * 100,  w, label='Adv Trained Fusion: Clean')
b4 = ax.bar(x + 1.5*w, adv_attack_pc * 100, w, label='Adv Trained Fusion: Conflict')

ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES, fontsize=12)
ax.set_ylabel('Per-class accuracy (%)')
ax.set_title(f'Fusion Per-Class Comparison at epsilon={EPS}\nSame Architecture, Same Split, Different Training Procedure',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim([0, 110])
ax.grid(axis='y', alpha=0.3)

for bars in [b1, b2, b3, b4]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                f'{h:.0f}%', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('/content/fusion_per_class_eps005.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFusion per-class comparison at epsilon={EPS}:')
print(f'{"Class":>12} | {"N":>5} | {"Std Clean":>9} | {"Std Attack":>10} | {"Adv Clean":>9} | {"Adv Attack":>10} | {"Gain":>8}')
print('-' * 82)

for i, name in enumerate(LABEL_NAMES):
    gain = adv_attack_pc[i] - std_attack_pc[i]
    print(f'{name:>12} | {int(class_totals[i]):>5} | {std_clean_pc[i]:>8.0%} | {std_attack_pc[i]:>9.0%} | {adv_clean_pc[i]:>8.0%} | {adv_attack_pc[i]:>9.0%} | {gain:>+8.0%}')

## 14. Attention Weight Analysis

Measures how much the LiDAR-sector attention pattern changes when the camera is attacked.

In [ ]:
# Collect attention weights for clean and attacked camera inputs
def collect_attention_matrix(model, loader, epsilon=0.0, attack=False):
    model.eval()
    all_attn = []
    all_correct = []
    all_labels = []

    for imgs, lids, lbls in loader:
        imgs, lids, lbls = imgs.to(device), lids.to(device), lbls.to(device)

        if attack and epsilon > 0:
            imgs = fgsm_attack_camera_only(model, imgs, lids, lbls, epsilon)

        with torch.no_grad():
            logits, attn = model(imgs, lids, return_attn=True)
            preds = logits.argmax(1)

        attn = attn.squeeze(1).detach().cpu().numpy()  # (B, 8)

        all_attn.append(attn)
        all_correct.extend((preds == lbls).cpu().numpy().tolist())
        all_labels.extend(lbls.cpu().numpy().tolist())

    return np.concatenate(all_attn, axis=0), np.array(all_correct), np.array(all_labels)


def attention_entropy(attn_matrix):
    return -(attn_matrix * np.log(attn_matrix + 1e-8)).sum(axis=1)


def attention_summary(name, clean_attn, conflict_attn):
    l1_shift = np.abs(conflict_attn - clean_attn).sum(axis=1).mean()
    entropy_clean = attention_entropy(clean_attn).mean()
    entropy_conflict = attention_entropy(conflict_attn).mean()
    top_sector_change = (clean_attn.argmax(axis=1) != conflict_attn.argmax(axis=1)).mean()

    print(f'\n{name}')
    print('-' * len(name))
    print(f'L1 attention shift         : {l1_shift:.4f}')
    print(f'Clean attention entropy    : {entropy_clean:.4f}')
    print(f'Conflict attention entropy : {entropy_conflict:.4f}')
    print(f'Top-sector change rate     : {top_sector_change:.2%}')
    print(f'Mean clean sector weights  : {np.round(clean_attn.mean(axis=0), 4)}')
    print(f'Mean attack sector weights : {np.round(conflict_attn.mean(axis=0), 4)}')

    return {
        'l1_shift': l1_shift,
        'entropy_clean': entropy_clean,
        'entropy_conflict': entropy_conflict,
        'top_sector_change': top_sector_change
    }


ATTACK_EPS = 0.05

print('Collecting attention weights...')

std_attn_clean, _, _ = collect_attention_matrix(
    standard_model,
    test_loader,
    attack=False
)

std_attn_conflict, _, _ = collect_attention_matrix(
    standard_model,
    test_loader,
    epsilon=ATTACK_EPS,
    attack=True
)

adv_attn_clean, _, _ = collect_attention_matrix(
    adv_model,
    test_loader,
    attack=False
)

adv_attn_conflict, _, _ = collect_attention_matrix(
    adv_model,
    test_loader,
    epsilon=ATTACK_EPS,
    attack=True
)

std_attn_stats = attention_summary(
    'Standard Fusion Attention',
    std_attn_clean,
    std_attn_conflict
)

adv_attn_stats = attention_summary(
    'Adversarially Trained Fusion Attention',
    adv_attn_clean,
    adv_attn_conflict
)

print('\nSector names:')
for i, name in enumerate(SECTOR_NAMES):
    print(f'  {i}: {name}')

## 15. Visualize Attention Shift

Plots average attention over the eight LiDAR sectors for clean and attacked camera inputs.

In [ ]:
# Plot average attention over the eight LiDAR sectors
std_sector_clean = std_attn_clean.mean(axis=0)
std_sector_conflict = std_attn_conflict.mean(axis=0)

adv_sector_clean = adv_attn_clean.mean(axis=0)
adv_sector_conflict = adv_attn_conflict.mean(axis=0)

x = np.arange(NUM_LIDAR_SECTORS)
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Standard fusion
axes[0].bar(x - w/2, std_sector_clean, w, label='Clean', color='steelblue', alpha=0.85)
axes[0].bar(x + w/2, std_sector_conflict, w, label='Conflict', color='firebrick', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(SECTOR_NAMES, rotation=35, ha='right', fontsize=8)
axes[0].set_title('Standard Fusion\nPer-Sector LiDAR Attention', fontweight='bold')
axes[0].set_ylabel('Attention weight')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xlabel(
    f"L1 shift: {std_attn_stats['l1_shift']:.4f} | "
    f"Top-sector change: {std_attn_stats['top_sector_change']:.1%}"
)

# Adversarially trained fusion
axes[1].bar(x - w/2, adv_sector_clean, w, label='Clean', color='steelblue', alpha=0.85)
axes[1].bar(x + w/2, adv_sector_conflict, w, label='Conflict', color='firebrick', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(SECTOR_NAMES, rotation=35, ha='right', fontsize=8)
axes[1].set_title('Adversarially Trained Fusion\nPer-Sector LiDAR Attention', fontweight='bold')
axes[1].set_ylabel('Attention weight')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_xlabel(
    f"L1 shift: {adv_attn_stats['l1_shift']:.4f} | "
    f"Top-sector change: {adv_attn_stats['top_sector_change']:.1%}"
)

plt.suptitle(
    'Cross-Attention Over Real-World LiDAR Sectors\nCamera Attacked, LiDAR Clean',
    fontsize=12,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('/content/attention_sector_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. Save Results

In [ ]:
# Save Cross-Attention Fusion results

import os
import shutil
import pickle
import zipfile

EXPERIMENT_NAME = "CrossAttention_Experiments"

# Try Google Drive first. If it fails, save locally and zip the results.
use_drive = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    SAVE_DIR = f"/content/drive/MyDrive/AV_Adversarial_Project/{EXPERIMENT_NAME}"
    use_drive = True
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Google Drive mount failed. Saving locally instead.")
    print("Error:", e)
    SAVE_DIR = f"/content/{EXPERIMENT_NAME}"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to: {SAVE_DIR}")

# Save model checkpoints and main result variables.
checkpoint = {
    "standard_model_state_dict": standard_model.state_dict(),
    "adv_model_state_dict": adv_model.state_dict(),
    "std_history": std_history,
    "adv_history": adv_history,
    "std_clean": std_clean,
    "std_conflict": std_conflict,
    "adv_clean": adv_clean,
    "adv_conflict": adv_conflict,
    "LABEL_NAMES": LABEL_NAMES,
    "SECTOR_NAMES": SECTOR_NAMES,
    "NUM_CLASSES": NUM_CLASSES,
    "NUM_LIDAR_SECTORS": NUM_LIDAR_SECTORS,
    "PTS_PER_SECTOR": PTS_PER_SECTOR,
    "EPSILONS": EPSILONS,
}

# Add optional outputs if those cells were run.
optional_vars = [
    "std_clean_pc",
    "std_attack_pc",
    "adv_clean_pc",
    "adv_attack_pc",
    "class_totals",
    "std_attn_stats",
    "adv_attn_stats",
]

for var_name in optional_vars:
    if var_name in globals():
        checkpoint[var_name] = globals()[var_name]

torch.save(checkpoint, os.path.join(SAVE_DIR, "crossattention_full_checkpoint.pth"))
print("Saved crossattention_full_checkpoint.pth")

# Save a smaller pickle file for the final comparison notebook.
results_to_save = {
    "std_clean": std_clean,
    "std_conflict": std_conflict,
    "adv_clean": adv_clean,
    "adv_conflict": adv_conflict,
    "std_history": std_history,
    "adv_history": adv_history,
    "LABEL_NAMES": LABEL_NAMES,
    "SECTOR_NAMES": SECTOR_NAMES,
    "EPSILONS": EPSILONS,
}

for var_name in optional_vars:
    if var_name in globals():
        results_to_save[var_name] = globals()[var_name]

with open(os.path.join(SAVE_DIR, "crossattention_results.pkl"), "wb") as f:
    pickle.dump(results_to_save, f)

print("Saved crossattention_results.pkl")

# Copy generated figures and videos if they exist.
files_to_try = [
    "training_comparison.png",
    "main_comparison.png",
    "fusion_per_class_eps005.png",
    "attention_sector_comparison.png",
    "fusion_per_class_eps005.png",
    "fusion_standard_demo.mp4",
    "fusion_adv_demo.mp4",
]

saved_files = []
for fname in files_to_try:
    src = os.path.join("/content", fname)
    dst = os.path.join(SAVE_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        saved_files.append(fname)

print("Saved generated files:")
if saved_files:
    for fname in saved_files:
        print(f"  {fname}")
else:
    print("  No generated files found yet.")

# Save a small text summary for the repo/Drive folder.
summary_path = os.path.join(SAVE_DIR, "save_summary.txt")
with open(summary_path, "w") as f:
    f.write(f"Experiment: {EXPERIMENT_NAME}\n")
    f.write(f"Save directory: {SAVE_DIR}\n\n")
    f.write("Saved files:\n")
    f.write("- crossattention_full_checkpoint.pth\n")
    f.write("- crossattention_results.pkl\n")
    for fname in saved_files:
        f.write(f"- {fname}\n")
    f.write("\nImportant variables saved:\n")
    for key in results_to_save.keys():
        f.write(f"- {key}\n")

print(f"Summary saved to: {summary_path}")

# If Drive was unavailable, create a zip that can be downloaded from Colab.
if not use_drive:
    zip_path = f"/content/{EXPERIMENT_NAME}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(SAVE_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, SAVE_DIR)
                zipf.write(file_path, arcname)
    print(f"Local ZIP created: {zip_path}")

print("Done saving Cross-Attention experiment.")
